# 01. Apollo Baseline

This notebook runs the full cross-validation benchmark on the purified tumor data.
It implements the exact methodology from the Apollo paper: Feature Selection (Top 20) + SVM.

In [1]:
import sys
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier

# Add project root to path so we can import 'src'
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.loader import GliomaDataLoader
from src.models.baselines import ApolloPipeline

In [2]:
# --- Configuration ---
DATA_PATH = "../data/processed/glioma_dataset.h5"
SPLIT_FILE = "../data/splits/manual_splits.json"

if not os.path.exists(SPLIT_FILE):
    print(f"❌ ERROR: Splits file not found at {SPLIT_FILE}")

print("Initializing Loader...")
# Ensure tumor_only=True so we only train on the data identified by our Spatio-Spectral Annotator
loader = GliomaDataLoader(
    data_path=DATA_PATH, 
    split_file=SPLIT_FILE, 
    memory_mapped=False, 
    tumor_only=True
)

print(f"✅ Loader initialized. Total Tumor Spectra: {loader.X.shape[0]}")

Initializing Loader...
📂 Loading data from ../data/processed/glioma_dataset.h5...
🔬 'tumor_only=True': Filtering out healthy tissue...
   ✅ Filtered to 189277 Pure Tumor Spectra.
✅ Loader initialized. Total Tumor Spectra: 189277


## Run Benchmark (5-Fold CV)

In [ ]:
fold_results = []
feature_masks = []

print("Starting Apollo Baseline Reproduction...")
print("-" * 60)

for fold_data in loader:
    fold_id = fold_data['fold_name']
    print(f"\n>>> Running {fold_id} <<<")
    
    # Unpack Data
    X_train, y_train = fold_data['train']
    X_val, y_val = fold_data['val']
    
    # Print fold stats
    print(f"    Train Samples: {X_train.shape[0]} | Val Samples: {X_val.shape[0]}")
    print(f"    Held-out Patients: {np.unique(fold_data['val_sample_ids'])}")
    
    # Initialize Model
    model = ApolloPipeline(n_features=20, random_state=42)
    
    # Train
    model.fit(X_train, y_train)
    
    # Save the selected features (Critical for Phase 3: TSTR)
    feature_masks.append(model.selected_indices_)
    
    # Predict
    y_pred = model.predict(X_val)
    
    # Evaluate
    acc = accuracy_score(y_val, y_pred)
    print(f"    > Fold Accuracy: {acc:.4f}")
    
    # Store detailed results
    fold_results.append({
        'fold': fold_id,
        'accuracy': acc,
        'y_true': y_val,
        'y_pred': y_pred
    })

print("-" * 60)
print("Benchmark Complete.")

## Analysis & Reporting

In [ ]:
# Aggregate Accuracy
accuracies = [r['accuracy'] for r in fold_results]
mean_acc = np.mean(accuracies)
std_acc = np.std(accuracies)

print(f"\n=== Final Results ===")
print(f"Apollo Baseline Accuracy: {mean_acc:.4f} ± {std_acc:.4f}")

# Create results directory if needed
os.makedirs("../results", exist_ok=True)

# Save simple metrics
metrics_df = pd.DataFrame(fold_results).drop(columns=['y_true', 'y_pred'])
metrics_df.to_csv("../results/baseline_metrics.csv", index=False)
print("✅ Saved clean metrics CSV.")

# Save feature masks for later TSTR evaluation
np.save("../results/feature_masks.npy", feature_masks)
print("✅ Saved selected feature masks to .npy")

In [ ]:
all_true = np.concatenate([r['y_true'] for r in fold_results])
all_pred = np.concatenate([r['y_pred'] for r in fold_results])

cm = confusion_matrix(all_true, all_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.title(f"Aggregated Confusion Matrix (Acc: {mean_acc:.4f})")
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.show()